# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Optional stretch, completed.** The card for this notebook was retired on 2026-07-13 and its core moved into the graded notebooks, but the capstone rewards the full-depth version.

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

The optional stretch notebook, done on the same ten development decision dates (2026-01-05 → 2026-05-11)
and with the same shared module (`work/scripts/search_queue.py`) as ML-09, ML-10 and the capstone. June 2026
stays sealed.

In [1]:
import os, sys, json, subprocess, importlib.util, urllib.request, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import numpy as np
import pandas as pd
import sklearn

# the shared module: from the repo when run inside it, else fetched from the repo (Colab opened from GitHub)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT / 'work' / 'scripts'))
else:
    ROOT = Path.cwd()
    urllib.request.urlretrieve('https://raw.githubusercontent.com/WingManOO7/ML-INTER/main/work/scripts/search_queue.py',
                               'search_queue.py')
import search_queue as sq

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

OUT_DIR = ROOT / 'work' / 'outputs'
CACHE = OUT_DIR / 'cache'            # parquet frames, gitignored: reruns skip the warehouse scans
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEV_DATES = sq.decision_dates('2026-01-05', '2026-05-11')     # every second Monday; outcomes end 2026-05-24
DEV_MONTHS = sq.months_needed(DEV_DATES)
assert '2026-06' not in DEV_MONTHS, 'June 2026 is the sealed month: development never opens it'

con = sq.open_warehouse(DEV_MONTHS, HF_TOKEN)
frames = sq.build_frames(con, DEV_DATES, cache_dir=CACHE)
dev = frames[frames['label_ok']].reset_index(drop=True)
SEED = 42

print(f'{len(DEV_DATES)} decision dates {DEV_DATES[0].date()} -> {DEV_DATES[-1].date()} | months {DEV_MONTHS[0]} -> {DEV_MONTHS[-1]}')
print(f'{len(frames):,} page-date rows; {len(dev):,} with a complete outcome window; '
      f'{dev.client_hash_id.nunique()} clients | pandas {pd.__version__}, scikit-learn {sklearn.__version__}')

10 decision dates 2026-01-05 -> 2026-05-11 | months 2025-12 -> 2026-05
659,482 page-date rows; 654,186 with a complete outcome window; 50 clients | pandas 2.3.3, scikit-learn 1.7.2


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
KEY = ['imp_pre', 'imp_wk1', 'imp_wk2', 'momentum_pre', 'pos_pre', 'active_days_pre', 'age_days_at_T', 'peak_day_share_pre']
q = dev[KEY].quantile([0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).T
q['max'] = dev[KEY].max()
top1 = dev['imp_pre'].sort_values(ascending=False)
share_top1 = top1.head(int(len(top1) * 0.01)).sum() / top1.sum()
print(f'impressions are heavy-tailed: median {dev.imp_pre.median():,.0f}, p99 {dev.imp_pre.quantile(0.99):,.0f}, '
      f'max {dev.imp_pre.max():,.0f}; the top 1% of page-dates hold {share_top1:.1%} of all impressions')
print(f"pages active on all 14 days: {(dev.active_days_pre == 14).mean():.1%}; position below 1 (impossible): {(dev.pos_pre < 1).mean():.1%}\n")
q.round(3)

impressions are heavy-tailed: median 512, p99 14,794, max 406,715; the top 1% of page-dates hold 17.3% of all impressions
pages active on all 14 days: 93.2%; position below 1 (impossible): 1.1%



,0.01,0.1,0.25,0.5,0.75,0.9,0.99,max
imp_pre,103.000,137.000,221.000,512.000,1370.000,3436.000,14794.000,406715.000
imp_wk1,23.000,65.000,108.000,253.000,682.000,1722.000,7469.150,296203.000
imp_wk2,30.000,64.000,107.000,249.000,679.000,1739.000,8028.150,241437.000
momentum_pre,-1.744,-0.582,-0.251,0.000,0.238,0.535,2.060,9.593
pos_pre,0.957,2.801,4.598,7.629,15.506,29.255,61.951,142.294
active_days_pre,9.000,14.000,14.000,14.000,14.000,14.000,14.000,14.000
age_days_at_T,13.000,41.000,87.000,181.000,272.000,369.000,461.000,535.000
peak_day_share_pre,0.085,0.096,0.106,0.122,0.148,0.190,0.393,1.000


**Heavy tails everywhere that matters:**
- **Impressions:** median **512** per page over 14 days, p99 14,794, maximum 406,715. The top 1% of page-dates hold **17.3%** of all impressions, which is why every model uses `log_imp_pre` and why a queue must weigh stake, not just probability.
- **The slide** (`momentum_pre`) is centred at 0, with fat tails both ways (p1 −1.74, p99 +2.06).
- **Most pages are steady:** 93.2% were active on all 14 days, and the median peak-day share is 0.12 (a perfectly flat page would score 1/14 ≈ 0.07).
- **Data quality:** 1.1% of page-dates carry an impossible position below 1.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each signal gets a bucket table of the forward two-week decline rate (`declined_robust`), plus a
**consistency check inside clients**. For every client-date with ≥ 200 pages, does the top bucket decline
more than the bottom one? A pooled trend that clients don't share is MIXED, not CONFIRMED.

- **#1 Recent slide** (`momentum_pre`). Expectation: pages already sliding keep sliding.
- **#2 Search position** (`pos_pre`). Expectation: deeper positions are less stable.
- **#3 Search volume** (`imp_pre`). Expectation: none in particular. Bigger pages *matter* more, but do they
  decline more?

In [3]:
def signal_test(col, edges, labels, higher_means):
    band = pd.cut(dev[col], edges, labels=labels, right=False)
    table = dev.groupby(band, observed=True)['declined_robust'].agg(pages='size', decline_rate='mean')
    lo, hi = labels[0], labels[-1]
    d = dev.assign(band=band)
    def gap(g):
        a, b = g.loc[g['band'] == hi, 'declined_robust'], g.loc[g['band'] == lo, 'declined_robust']
        return a.mean() - b.mean() if len(a) >= 20 and len(b) >= 20 else np.nan
    cells = d.groupby(['decision_date', 'client_hash_id']).filter(lambda g: len(g) >= 200)
    gaps = cells.groupby(['decision_date', 'client_hash_id']).apply(gap, include_groups=False).dropna()
    agree = (gaps > 0).mean() if higher_means == 'more' else (gaps < 0).mean()
    return table, gaps, agree

tests = {
    '#1 slide (momentum_pre)': signal_test('momentum_pre', [-np.inf, np.log(0.5), np.log(0.8), np.log(1.25), np.log(2), np.inf],
                                           ['<0.5x', '0.5-0.8x', '~1x', '1.25-2x', '>2x'], 'less'),
    '#2 position (pos_pre)': signal_test('pos_pre', [0, 3, 10, 20, 50, np.inf], ['top 3', '3-10', '10-20', '20-50', '50+'], 'more'),
    '#3 volume (imp_pre)': signal_test('imp_pre', [100, 300, 1000, 3000, 10000, np.inf], ['100-300', '300-1k', '1k-3k', '3k-10k', '10k+'], 'more'),
}
for name, (table, gaps, agree) in tests.items():
    print(f'{name}: inside-client cells agreeing with the expected direction (top vs bottom bucket): {agree:.0%} of {len(gaps)}')
    print(table.round(3).to_string(), '\n')

#1 slide (momentum_pre): inside-client cells agreeing with the expected direction (top vs bottom bucket): 98% of 141
               pages  decline_rate
momentum_pre                      
<0.5x          48948         0.719
0.5-0.8x      127993         0.472
~1x           304556         0.297
1.25-2x       128767         0.204
>2x            43922         0.232 

#2 position (pos_pre): inside-client cells agreeing with the expected direction (top vs bottom bucket): 53% of 73
          pages  decline_rate
pos_pre                      
top 3     75099         0.342
3-10     329053         0.321
10-20    128142         0.326
20-50    107543         0.390
50+       14349         0.505 

#3 volume (imp_pre): inside-client cells agreeing with the expected direction (top vs bottom bucket): 44% of 79
          pages  decline_rate
imp_pre                      
100-300  224545         0.366
300-1k   218642         0.333
1k-3k    134018         0.317
3k-10k    62870         0.313
10k+      14111   

| Signal | Pooled pattern | Inside clients | Verdict |
|---|---|---|---|
| **#1 Recent slide** | 0.719 for pages that collapsed below half, falling to 0.204 for pages up 25–100% (0.232 above 2×) | the top bucket declines less than the bottom in **98%** of 141 client-dates | **CONFIRMED** |
| **#2 Position** | flat from the top 3 to position 20 (0.32–0.34), then 0.390 at 20–50 and 0.505 past 50 | deep pages decline more in only **53%** of 73 client-dates | **MIXED**: real for the deep tail pooled, not consistent inside sites |
| **#3 Volume** | U-shaped: 0.366 → 0.313 → 0.365 | bigger pages decline more in **44%** of 79 client-dates | **FALSE**: size doesn't predict decline |

Only the slide survives both readings, which is why it carries the ML-07 rule and the model (`momentum_pre` is the model's dominant feature, ML-08). Volume earns its place in the capstone policy as the *stake* (how much a decline costs), not as a predictor.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**The flag:** FlyRank's **refresh flags** assume that *staleness* predicts decline: old or long-unrefreshed
content drifts down and should be refreshed first. There are two ways to measure staleness here.

- **Days since the last update** is the real input of a refresh flag. It **can't be tested honestly** in
  this release: `content_updated_date` is a July snapshot, and for most pages it lies after T (ML-04,
  ML-09).
- **Page age** (T − creation date) is the closest honest proxy. The test below reads it pooled, inside
  clients, and **among pages with real visibility** (≥ 1,000 impressions), where a refresh decision would
  actually be made.

In [4]:
age_edges, age_labels = [0, 90, 180, 270, 365, np.inf], ['0-89 d', '90-179 d', '180-269 d', '270-364 d', '365+ d']
pooled_table, age_gaps, age_agree = signal_test('age_days_at_T', age_edges, age_labels, 'more')
big = dev[dev['imp_pre'] >= 1000]
big_table = big.groupby(pd.cut(big['age_days_at_T'], age_edges, labels=age_labels, right=False), observed=True)['declined_robust'] \
               .agg(pages='size', decline_rate='mean')
print(f'inside-client cells where the oldest pages (365+ d) decline more than the youngest (<90 d): {age_agree:.0%} of {len(age_gaps)}\n')
print('all eligible pages:'); print(pooled_table.round(3).to_string(), '\n')
print('pages with >= 1,000 impressions:'); print(big_table.round(3).to_string())

inside-client cells where the oldest pages (365+ d) decline more than the youngest (<90 d): 24% of 21

all eligible pages:
                pages  decline_rate
age_days_at_T                      
0-89 d         169464         0.367
90-179 d       153583         0.325
180-269 d      161458         0.352
270-364 d      101597         0.311
365+ d          68084         0.321 

pages with >= 1,000 impressions:
               pages  decline_rate
age_days_at_T                     
0-89 d         50937         0.330
90-179 d       52564         0.307
180-269 d      58535         0.349
270-364 d      30060         0.277
365+ d         18903         0.301


**Verdict: OPPOSITE (weak).** The data runs against the refresh flag's assumption.
- **Pooled:** the youngest pages decline most (0.367 for 0–89 days) and the oldest least (0.321 at 365+ days).
- **Large pages only** (≥ 1,000 impressions, where a refresh decision is actually made): no rise with age either (0.330 → 0.301).
- **Inside clients:** the oldest pages declined more than the youngest in only **24%** of 21 client-dates.

The effect is small, so "older pages are safer" would be too strong. But "older pages decline more", the premise behind age-based refresh flags, is not supported for a two-week horizon in this release. The real refresh input, days since the last *update*, can't be tested here, because the warehouse only has a July snapshot of it.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Watch the slide, not the calendar.** A page that already lost impressions from one week to the next is the best observed predictor that it will stay down; inside 98% of client-dates, collapsed pages declined more than rising ones. Page age and page size don't separate decliners. Size still matters, but as the stake that decides which declines are worth an editor's hour. **So schedule reviews from observed slides weighted by visibility, and treat age-based refresh calendars as a hypothesis to test** (ideally with real editorial update dates and a comparison group), not as a rule. These are observational, decision-support findings for one pseudonymized portfolio, January–May 2026.

In [5]:
receipts = {'notebook': 'work/notebooks/w04_signal_audit.ipynb',
            'signals': {name: {'table': t.round(4).reset_index().astype(str).to_dict('records'), 'inside_client_agree': round(float(a), 4),
                               'cells': int(len(g))} for name, (t, g, a) in tests.items()},
            'flag_linked_age': {'pooled': pooled_table.round(4).reset_index().astype(str).to_dict('records'),
                                'big_pages': big_table.round(4).reset_index().astype(str).to_dict('records'),
                                'inside_client_agree': round(float(age_agree), 4)}}
(OUT_DIR / 'w04x_signal_audit_receipts.json').write_text(json.dumps(receipts, indent=2))
print('wrote', (OUT_DIR / 'w04x_signal_audit_receipts.json').relative_to(ROOT))

wrote work\outputs\w04x_signal_audit_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.